In [0]:
%sql
use catalog capstone_team1;
use schema bronze;

In [0]:
primary_df=spark.table("capstone_team1.bronze.bronze_primary_events")
secondary_df=spark.table("capstone_team1.bronze.bronze_secondary_events")
status_df=spark.table("capstone_team1.bronze.bronze_status_updates")
primary_df.printSchema()
secondary_df.printSchema()
status_df.printSchema()

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, IntegerType
from pyspark.sql import functions as F

# Define explicit schemas
primary_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("ingest_timestamp", StringType(), True),
    StructField("merchant_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("payment_channel", StringType(), True),
    StructField("payment_status", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("risk_score", DoubleType(), True),
    StructField("device_id", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("source_schema_version", StringType(), True)
])

secondary_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("transaction_id", StringType(), True),
    StructField("event_timestamp", TimestampType(), True),
    StructField("related_entity_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("event_value", DoubleType(), True),
    StructField("source_system", StringType(), True)
])

status_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("change_timestamp", TimestampType(), True),
    StructField("previous_status", StringType(), True),
    StructField("new_status", StringType(), True),
    StructField("change_reason", StringType(), True),
    StructField("source_sequence", IntegerType(), True),
    StructField("source_system", StringType(), True)
])

# Cast all columns in each DataFrame based on the explicit schemas
def cast_to_schema(df, schema):
    for field in schema.fields:
        if field.name in df.columns:
            df = df.withColumn(field.name, F.col(field.name).cast(field.dataType))
    return df

primary_df = cast_to_schema(primary_df, primary_schema)
secondary_df = cast_to_schema(secondary_df, secondary_schema)
status_df = cast_to_schema(status_df, status_schema)

# Verify the casted schemas
print("Primary Events Schema:")
primary_df.printSchema()
print("Secondary Events Schema:")
secondary_df.printSchema()
print("Status Updates Schema:")
status_df.printSchema()

In [0]:
print("Primary events")
print(primary_df.columns)
print("\nSecondary events")
print(secondary_df.columns)
print("\nStatus updates")
print(status_df.columns)    

In [0]:
# Data Quality Rule Catalogue

dq_rules = [
    # Primary Events
    ("primary_events", "transaction_id", "NOT NULL"),
    ("primary_events", "event_timestamp", "VALID TIMESTAMP"),
    ("primary_events", "ingest_timestamp", "VALID TIMESTAMP"),
    ("primary_events", "merchant_id", "NOT NULL"),
    ("primary_events", "customer_id", "NOT NULL"),
    ("primary_events", "amount", "MUST BE POSITIVE"),
    ("primary_events", "risk_score", "BETWEEN 0 AND 100"),
    ("primary_events", "payment_status", "NOT NULL"),
    ("primary_events", "currency", "NOT NULL"),

    # Secondary Events
    ("secondary_events", "event_id", "NOT NULL"),
    ("secondary_events", "transaction_id", "NOT NULL"),
    ("secondary_events", "event_timestamp", "VALID TIMESTAMP"),
    ("secondary_events", "event_type", "NOT NULL"),

    # Status Updates
    ("status_updates", "transaction_id", "NOT NULL"),
    ("status_updates", "change_timestamp", "VALID TIMESTAMP"),
    ("status_updates", "new_status", "NOT NULL"),
    ("status_updates", "change_reason", "NOT NULL"),
]

# Convert rules into a DataFrame
dq_catalogue = spark.createDataFrame(
    dq_rules,
    ["dataset", "column_name", "dq_rule"]
)

display(dq_catalogue)

In [0]:
#verify Catalogue
print("Total DQ Rules:", dq_catalogue.count())
dq_catalogue.groupBy("dataset").count().show()